# Day 12 — CuTe DSL Elementwise Add

**Goal:** `C = A + B` for FP16 matrices, first naively
(1 element / thread) then vectorized (8 elements / thread → one
128-bit transaction). Memory-bound problem — coalescing and
vectorization are the levers.


In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch

cutlass.cuda.initialize_cuda_context()

M, N = 512, 1024
a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros_like(a)
a_, b_, c_ = (from_dlpack(t, assumed_align=16) for t in (a, b, c))


## Part 1 — naive: 1 element per thread

Each thread computes its global linear id, converts to `(mi, ni)` with
`ni` varying fastest (so the warp coalesces on N), and does one scalar
load + store. Guard with `if mi < m:`.


In [ ]:
@cute.kernel
def naive_kernel(gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor):
    # TODO: implement scalar 1-element-per-thread elementwise add.
    raise NotImplementedError


@cute.jit
def naive_elementwise_add(mA, mB, mC):
    threads_per_block: cutlass.Constexpr = 256
    n_elem = cute.size(mA)
    grid = (n_elem + threads_per_block - 1) // threads_per_block
    naive_kernel(mA, mB, mC).launch(
        grid=(grid, 1, 1), block=(threads_per_block, 1, 1)
    )


# naive_elementwise_add(a_, b_, c_)
# torch.cuda.synchronize()
# torch.testing.assert_close(c, a + b); print("naive OK")


## Part 2 — vectorized: 8 elements per thread

`cute.zipped_divide(mA, (1, 8))` reshapes `(M, N)` into
`((1, 8), (M, N//8))`. The inner mode is the per-thread tile; outer mode
is the per-thread index. Then:

```python
a_vec = gA[(None, (mi, ni))].load()  # one ld.global.v8.f16
gC[(None, (mi, ni))] = a_vec + b_vec
```


In [ ]:
@cute.kernel
def vectorized_kernel(gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor):
    # TODO: each thread loads/stores a (1,8) sub-tile via .load() / assignment.
    raise NotImplementedError


@cute.jit
def vectorized_elementwise_add(mA, mB, mC):
    threads_per_block: cutlass.Constexpr = 256
    gA = cute.zipped_divide(mA, (1, 8))
    gB = cute.zipped_divide(mB, (1, 8))
    gC = cute.zipped_divide(mC, (1, 8))
    num_tiles = cute.size(gC, mode=[1])
    grid = (num_tiles + threads_per_block - 1) // threads_per_block
    vectorized_kernel(gA, gB, gC).launch(
        grid=(grid, 1, 1), block=(threads_per_block, 1, 1)
    )


# c.zero_(); c_ = from_dlpack(c, assumed_align=16)
# vectorized_elementwise_add(a_, b_, c_)
# torch.cuda.synchronize()
# torch.testing.assert_close(c, a + b); print("vectorized OK")


---

## Reference solution


In [ ]:
@cute.kernel
def naive_kernel(gA, gB, gC):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    gid = bidx * bdim + tidx
    m, n = gA.shape
    mi = gid // n
    ni = gid % n
    if mi < m:
        gC[mi, ni] = gA[mi, ni] + gB[mi, ni]


@cute.kernel
def vectorized_kernel(gA, gB, gC):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    thread_id = bidx * bdim + tidx
    m, n = gA.shape[1]
    mi = thread_id // n
    ni = thread_id % n
    a_vec = gA[(None, (mi, ni))].load()
    b_vec = gB[(None, (mi, ni))].load()
    gC[(None, (mi, ni))] = a_vec + b_vec


c.zero_(); c_ = from_dlpack(c, assumed_align=16)
naive_elementwise_add(a_, b_, c_)
torch.cuda.synchronize()
torch.testing.assert_close(c, a + b); print("naive OK")

c.zero_(); c_ = from_dlpack(c, assumed_align=16)
vectorized_elementwise_add(a_, b_, c_)
torch.cuda.synchronize()
torch.testing.assert_close(c, a + b); print("vectorized OK")
